# 03 — Hyperparameter Tuning BiLSTM (Lengkap)

Notebook ini melakukan pencarian hyperparameter secara sistematis. Inilah
bagian yang menjawab kritik reviewer: **bukan satu resep, tetapi ruang
pencarian yang jelas + protokol seleksi yang jujur.**

## Protokol

1. HPO hanya memakai **train + validation**.
2. **Frozen holdout tidak disentuh** sampai model terbaik dipilih.
3. Setiap trial dicatat: parameter, metrik, durasi, status.
4. Model dipilih berdasarkan metrik validation (default F1).
5. Setelah terbaik dipilih, model di-retrain dan diuji **sekali** di holdout.
6. Untuk paper, jalankan beberapa seed dan laporkan varians.

## Ruang pencarian

Preset:

- `smoke` — 4 trial, untuk memastikan pipeline jalan.
- `standard` — 30 trial, disarankan untuk paper.
- `full` — 100+ trial, eksplorasi maksimal.

Parameter yang dicari mencakup:

- **Vektorisasi**: vocab, max_len, dimensi embedding, window, min_count,
  epoch Word2Vec, CBOW/skip-gram, trainable embedding, mask_zero, digit_fold.
- **Arsitektur**: LSTM/GRU, bidirectional/tidak, 1–2 lapisan, unit, dropout,
  recurrent dropout, dense, MaxNorm.
- **Optimasi**: Adam/AdamW/RMSprop/SGD, learning rate, weight decay, batch,
  gradient clip, class weight, label smoothing.
- **Callback**: patience early stopping, patience/factor reduce LR.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook biasanya dijalankan dari folder notebooks/. Kita pindah ke root
# project agar semua path artefak konsisten.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "artifacts" / "data"
BERT_DIR = PROJECT_ROOT / "artifacts" / "bert" / "input"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "notebook_runs"
for path in (DATA_DIR, BERT_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("BERT input   :", BERT_DIR)
print("Runs dir     :", RUNS_DIR)

In [ ]:
from app.experiment.configs import DataConfig
from app.experiment.data_pipeline import prepare_data

config_path = DATA_DIR / "data_config.json"
if config_path.exists():
    data_config = DataConfig.from_dict(json.loads(config_path.read_text(encoding="utf-8")))
else:
    candidates = [
        PROJECT_ROOT / "ocr_output" / "data-mengandung.csv",
        PROJECT_ROOT.parent / "old" / "ocr_output" / "data-mengandung.csv",
        PROJECT_ROOT / "data.csv",
    ]
    csv_path = next((path for path in candidates if path.exists()), None)
    if csv_path is None:
        raise FileNotFoundError(
            "CSV tidak ditemukan. Set DataConfig.csv_input secara manual "
            "ke file dengan kolom: nama_produk;text;label"
        )
    data_config = DataConfig(
        csv_input=str(csv_path),
        output_dir=str(DATA_DIR),
        bert_input_dir=str(BERT_DIR),
        sort_source=True,
    )

# prepare_data bersifat deterministik dan cepat; aman dijalankan ulang.
bundle = prepare_data(data_config)
print(bundle.summary_frame().to_string(index=False))

In [ ]:
from app.experiment.configs import BiLSTMConfig, HPOConfig
from app.experiment.search_spaces import bilstm_search_space, preset_trials, space_summary
from app.experiment.hpo import HPOEngine
from app.experiment.bilstm_pipeline import build_vectorizers, run_bilstm_trial, train_final_bilstm

PRESET = "smoke"          # ganti ke "standard" atau "full" untuk paper
N_TRIALS = preset_trials(PRESET)
space = bilstm_search_space(PRESET)
print(f"Preset={PRESET}, target trial={N_TRIALS}, jumlah parameter={len(space.params)}")
pd.DataFrame(space_summary(space))

## Fungsi objective

Satu trial = satu konfigurasi. Fungsi ini:

1. mengubah dict parameter menjadi `BiLSTMConfig` (validasi otomatis),
2. memakai cache vektorisasi bila parameter Word2Vec sama,
3. melatih model,
4. mengembalikan metrik validation.

Holdout tidak dipanggil di sini.

In [ ]:
from app.experiment.bilstm_pipeline import vectorizer_key

_vector_cache = {}

def objective(params, trial_index, trial_dir):
    config = BiLSTMConfig.from_dict(params)
    key = vectorizer_key(config)
    vectorized = _vector_cache.get(key)
    if vectorized is None:
        # Cache dibatasi agar memori tidak membengkak saat banyak trial.
        if len(_vector_cache) >= 2:
            _vector_cache.clear()
        vectorized = build_vectorizers(bundle, config)
        _vector_cache[key] = vectorized
    result = run_bilstm_trial(
        bundle,
        config,
        trial_dir,
        vectorized=vectorized,
        save_model=False,
        save_predictions=False,
        evaluate_holdout=False,
        verbose=0,
    )
    return {"metrics": result.metrics, "artifacts": result.artifacts}

## Jalankan HPO

`timeout_minutes` membatasi waktu. `resume=True` membuat notebook bisa
dilanjutkan tanpa mengulang trial yang sudah sukses.

In [ ]:
HPO_DIR = RUNS_DIR / "bilstm_hpo"
hpo_config = HPOConfig(
    strategy="random",          # coba juga "grid" (jika grid kecil) atau "bayesian"
    n_trials=N_TRIALS,
    seed=42,
    timeout_minutes=120,
    objective_metric="val_f1",
    objective_mode="max",
    output_dir=str(HPO_DIR),
    study_name=f"bilstm_{PRESET}",
    resume=True,
    save_trial_models=False,
)

engine = HPOEngine(hpo_config, space, run_dir=HPO_DIR / f"{PRESET}_seed42", verbose=True)
hpo_result = engine.run(objective)
print("Run dir:", hpo_result.run_dir)
if hpo_result.best is None:
    raise RuntimeError("Tidak ada trial sukses. Cek error di trials.jsonl.")

## Leaderboard

Semua trial diurutkan berdasarkan objective. Perhatikan bahwa trial yang
gagal tetap dicatat — ini penting untuk audit.

In [ ]:
leaderboard = pd.read_csv(hpo_result.run_dir / "leaderboard.csv")
columns = ["trial_id", "status", "objective", "metric_val_f1", "metric_val_roc_auc", "metric_val_loss", "duration_sec"]
columns = [column for column in columns if column in leaderboard.columns]
leaderboard[columns].head(15)

In [ ]:
best = hpo_result.best
print("Best trial :", best.trial_id)
print("Best params:")
print(json.dumps(best.params, indent=2, ensure_ascii=False))
print("\nBest metrics:")
print(json.dumps(best.metrics, indent=2, ensure_ascii=False))

## Plot progres dan importance

- **Progres**: apakah pencarian menemukan perbaikan dari trial ke trial?
- **Importance**: parameter mana yang paling berpengaruh? Ini bahan bagus
  untuk diskusi paper.

In [ ]:
from app.experiment.reporting import plot_hpo_progress, plot_param_importance

plot_hpo_progress(
    leaderboard,
    hpo_result.run_dir / "hpo_progress.png",
    mode=hpo_config.objective_mode,
)
plot_param_importance(leaderboard, hpo_result.run_dir / "hpo_importance.png")
print("Plot tersimpan di", hpo_result.run_dir)

## Retrain model terbaik dan uji holdout sekali

Hyperparameter terbaik dari validation sekarang dilatih ulang pada
final-train, lalu diuji di frozen holdout. Ini satu-satunya saat holdout
dipakai.

In [ ]:
best_config = BiLSTMConfig.from_dict(best.params)
BEST_OUT = RUNS_DIR / "bilstm_tuned"
BEST_MODEL = BEST_OUT / "models"

best_output = train_final_bilstm(
    bundle,
    best_config,
    output_dir=BEST_OUT,
    model_dir=BEST_MODEL,
    hpo_info={
        "source": str(hpo_result.run_dir),
        "best_trial": best.trial_id,
        "best_params": best.params,
        "best_metrics": best.metrics,
        "search_space": space.to_dict(),
        "n_trials": len(hpo_result.trials),
    },
    verbose=1,
)
print(best_output["eval_table"].to_string(index=False))

## Bandingkan baseline vs tuned

Kalau notebook 02 sudah dijalankan, kita bisa membandingkan pada holdout yang
sama memakai uji McNemar + paired bootstrap AUC.

In [ ]:
from app.experiment.reporting import compare_two_models, load_probabilities

baseline_path = RUNS_DIR / "bilstm_baseline" / "probs_holdout.csv"
if baseline_path.exists():
    y_base, p_base, _ = load_probabilities(baseline_path)
    comparison = compare_two_models(
        y_base, p_base, best_output["result"].holdout_prob,
        name_a="BiLSTM baseline", name_b="BiLSTM tuned",
        n_bootstrap=2000, seed=42,
    )
    print(json.dumps(comparison, indent=2, ensure_ascii=False))
else:
    print("Jalankan notebook 02 dulu untuk perbandingan baseline.")

## Cara melaporkan HPO di paper

Sertakan minimal:

1. **Search space** (tabel parameter + rentang/nilai).
2. **Jumlah trial** dan strategi (grid/random/Bayesian).
3. **Metrik seleksi** (mis. F1 validation) dan arahnya.
4. **Protokol holdout**: holdout tidak dipakai untuk tuning.
5. **Seed** dan varians antar run.
6. **Uji statistik** baseline vs tuned.
7. **Ablasi** synthetic vs real-only (set `synthetic_total=0`).

Contoh kalimat:

> Pencarian hiperparameter dilakukan dengan random search sebanyak 30 trial
> pada ruang 33 parameter (Tabel X). Model dipilih berdasarkan F1 pada
> validation set; frozen holdout (n=50) hanya diprediksi sekali setelah
> konfigurasi terbaik dibekukan. Perbedaan performa terhadap baseline diuji
> dengan McNemar dan paired bootstrap (2.000 resampling).

## Peringatan overfitting validation

Validation hanya 50 sampel. Jika Anda menjalankan 100+ trial dan memilih yang
terbaik, sebagian keunggulan bisa berasal dari keberuntungan. Untuk paper:

- jalankan minimal 3 seed,
- laporkan mean ± std,
- pertimbangkan repeated group-aware cross-validation,
- jangan mengubah holdout untuk "memperbaiki" hasil.